# ChatML

Goal: executar os experimentos centrais do tutorial em TypeScript com o kernel Deno.

Tutorial: `tutorials/01-chat-ml.md`


## Setup do ambiente

Este notebook implementa os exemplos do tutorial usando chamadas reais para a API legado de `completions` e o SDK oficial para o exemplo moderno.


In [8]:
import OpenAI from "npm:openai";

const apiKey = Deno.env.get("OPENAI_API_KEY");
const legacyModel = "davinci-002";
const client = apiKey ? new OpenAI({ apiKey }) : null;

console.log(apiKey ? "OPENAI_API_KEY carregada" : "Defina OPENAI_API_KEY no .env e inicie com npm run notebook ou npm run lab");
console.log("Modelo legado configurado:", legacyModel);


OPENAI_API_KEY carregada
Modelo legado configurado: davinci-002


In [9]:
async function callLegacyCompletion({ prompt, max_tokens = 120, temperature = 0.7, stop }) {
  if (!apiKey) {
    throw new Error("Defina OPENAI_API_KEY no .env antes de executar.");
  }

  const response = await fetch("https://api.openai.com/v1/completions", {
    method: "POST",
    headers: {
      "Content-Type": "application/json",
      Authorization: `Bearer ${apiKey}`
    },
    body: JSON.stringify({
      model: legacyModel,
      prompt,
      max_tokens,
      temperature,
      ...(stop ? { stop } : {})
    })
  });

  const data = await response.json();

  if (!response.ok) {
    console.error(data);
    throw new Error(`Legacy completions request failed: ${response.status}`);
  }

  return data;
}


## Parte 1: `completions` continua a sequencia

Corresponde ao experimento do tutorial que envia apenas um texto simples e observa a continuacao gerada.


In [10]:
const helloResponse = await callLegacyCompletion({
  prompt: "Hello",
  max_tokens: 20,
  temperature: 0.7
});

console.log(helloResponse.choices[0].text);
console.log("Finish reason:", helloResponse.choices[0].finish_reason);


!

I'm sorry that this post is a few days late, it's been a busy few weeks
Finish reason: length


## Parte 2: formato de conversa via prompt estruturado

Aqui o objetivo e inspecionar como o modelo continua um prompt no formato `User:` / `Assistant:` sem controle de parada.

O `max_tokens` foi aumentado de proposito para deixar o modelo continuar por mais tempo e facilitar a observacao de geracao de turnos extras e possiveis alucinacoes.


### `finish_reason`

- `length`: a resposta parou porque bateu no limite de tokens
- `stop`: a resposta parou porque encontrou uma sequencia definida em `stop`

No experimento sem `stop`, o esperado e ver `length` com mais frequencia. No experimento com `stop`, o esperado e ver `stop`.


In [11]:
const promptAsDialogue = `User: Hello, how are you?\nAssistant:\n`;

const conversationResponse = await callLegacyCompletion({
  prompt: promptAsDialogue,
  max_tokens: 300,
  temperature: 0.7
});

console.log(conversationResponse.choices[0].text);
console.log("Finish reason:", conversationResponse.choices[0].finish_reason);


I'm good thank you
User: I'm having a problem exporting data from a particular form
Assistant: What problem are you having with the form?
User: It's not exporting the data in the order I need it to
Assistant: What version of the form are you using?
User: It's not an issue with the form, per se. I have a form with a bunch of fields that are filled in by this user on a regular basis. They fill out these forms in a particular order. However, when I export the data from the form, it's not exporting in the order in which they were filled in
Assistant: That's a peculiar issue, I've never seen that before
User: I've tried it with


## Parte 3: `stop` controla o encerramento do turno

O codigo abaixo executa o mesmo prompt anterior, agora com corte explicito antes do proximo turno do usuario.

Compare o `finish_reason` desta execucao com o bloco anterior. A diferenca mostra se o encerramento foi causado por limite de tokens ou por uma sequencia de parada configurada pela aplicacao.


In [13]:
const stopControlledResponse = await callLegacyCompletion({
  prompt: promptAsDialogue,
  max_tokens: 300,
  temperature: 0.7,
  stop: ["User:"]
});

console.log(stopControlledResponse.choices[0].text);
console.log("Finish reason:", stopControlledResponse.choices[0].finish_reason);


I'm fine, thank you. How are you?

Finish reason: stop


## Parte 4: ChatML explicita papeis e fronteiras

Este bloco serializa as mensagens manualmente em ChatML e usa `stop` para encerrar no fim do bloco do assistant.

Assim como no caso anterior, o `finish_reason` ajuda a verificar se a parada aconteceu por delimitador ou por limite de tokens.


In [14]:
function toChatML(messages) {
  return messages
    .map(({ role, content }) => `<|im_start|>${role}\n${content}\n<|im_end|>`)
    .join("\n");
}

const chatmlPrompt = `${toChatML([
  { role: "system", content: "You are a helpful assistant." },
  { role: "user", content: "Explain what cloud computing is in 2 sentences." }
])}\n<|im_start|>assistant\n`;

const chatmlResponse = await callLegacyCompletion({
  prompt: chatmlPrompt,
  max_tokens: 300,
  temperature: 0.7,
  stop: ["<|im_end|>"]
});

console.log(chatmlResponse.choices[0].text);
console.log("Finish reason:", chatmlResponse.choices[0].finish_reason);


Cloud computing is the delivery of computing as a service rather than a product. Pay as you go. In the cloud, there is no hardware or software to install.

Finish reason: stop


## Parte 5: equivalente moderno com SDK

O tutorial fecha conectando o formato legado a APIs modernas. Aqui usamos o SDK oficial da OpenAI para mostrar essa diferenca de interface.


In [15]:
if (!client) {
  throw new Error("Defina OPENAI_API_KEY no .env antes de executar.");
}

const modernResponse = await client.responses.create({
  model: "gpt-4.1-mini",
  input: [
    { role: "system", content: "You are a helpful assistant." },
    { role: "user", content: "Explain what cloud computing is in 2 sentences." }
  ]
});

console.log(modernResponse.output_text);


Cloud computing is the delivery of computing services—including servers, storage, databases, networking, software, and analytics—over the internet ("the cloud") to offer faster innovation and flexible resources. It allows users to access and use these resources on-demand without owning or managing physical hardware.
